# Mr. Chicken + MuseTalk 1.5 Oficial no Kaggle/Colab

Este notebook executa o **MuseTalk 1.5 Oficial** (sincronização labial em tempo real da Tencent Music Lyra Lab) como um microserviço REST FastAPI, compatível com o Mr. Chicken local.


In [ ]:
# 1. Configurações de Ambiente e Pastas
from pathlib import Path
import os, secrets, sys, time, socket

def check_internet():
    try:
        socket.setdefaulttimeout(3)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(('8.8.8.8', 53))
        return True
    except socket.error:
        return False

if not check_internet():
    print('========================================================================')
    print('AVISO: Sem conexão com a internet detectada!')
    print('Se você estiver no Kaggle, certifique-se de ativar a opção "Internet"')
    print('no painel lateral direito (Settings -> Internet -> deslize para ON).')
    print('========================================================================')
else:
    print('Conexão com a internet: OK')

WORK_DIR = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
REPO_DIR = WORK_DIR / 'MuseTalk'
OUTPUTS_DIR = WORK_DIR / 'mrchicken_lipsync_outputs'
PORT = int(os.environ.get('LIPSYNC_PORT', '8010'))

# MuseTalk/OpenMMLab ? sens?vel ? vers?o: Python 3.10 + Torch 2.0.1/cu118 evita builds lentos/quebrados de mmcv.
import subprocess
ENV_DIR = WORK_DIR / 'musetalk-py310'
def python_version(exe):
    try:
        result = subprocess.run([exe, '-c', 'import sys; print(f"{sys.version_info.major}.{sys.version_info.minor}")'], capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except Exception:
        return ''

base_python = '/opt/conda/bin/python' if os.path.exists('/opt/conda/bin/python') else sys.executable
if python_version(base_python) == '3.10':
    PYTHON_EXE = base_python
elif not ENV_DIR.exists():
    print('Criando ambiente Python 3.10 isolado com micromamba...')
    subprocess.run(['bash', '-lc', f'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C {WORK_DIR} bin/micromamba'], check=True)
    micromamba = WORK_DIR / 'bin' / 'micromamba'
    subprocess.run([str(micromamba), 'create', '-y', '-p', str(ENV_DIR), '-c', 'conda-forge', 'python=3.10', 'pip'], check=True)
    PYTHON_EXE = str(ENV_DIR / 'bin' / 'python')
else:
    PYTHON_EXE = str(ENV_DIR / 'bin' / 'python')
os.environ['PYTHON'] = PYTHON_EXE
os.environ['MPLBACKEND'] = 'Agg'

# API Key configurada para segurança (deve bater com o LIPSYNC_API_KEY no Next.js)
API_KEY = "13991059620"
os.environ['LIPSYNC_API_KEY'] = API_KEY
os.environ['MUSETALK_REPO_PATH'] = str(REPO_DIR)
os.environ['MUSETALK_OUTPUTS_DIR'] = str(OUTPUTS_DIR)
os.environ['MUSETALK_TIMEOUT_SECONDS'] = '1800'
os.environ['HF_HOME'] = str(WORK_DIR / 'hf-cache')

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print('WORK_DIR=', WORK_DIR)
print('REPO_DIR=', REPO_DIR)
print('OUTPUTS_DIR=', OUTPUTS_DIR)
print('PORT=', PORT)
print('PYTHON_EXE=', PYTHON_EXE)


: 

In [ ]:
# 2. Clone/update do reposit?rio oficial MuseTalk
import shutil, subprocess, sys, os

giturl = 'https://github.com/TMElyralab/MuseTalk.git'

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    print("Diret?rio MuseTalk existe mas n?o ? reposit?rio git. Removendo...")
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    print("Clonando reposit?rio MuseTalk oficial...")
    subprocess.run(['git', 'clone', giturl, str(REPO_DIR)], check=True)
else:
    print("Reposit?rio MuseTalk j? existe. Efetuando git pull...")
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

try:
    print("Verificando/instalando pacotes de sistema necess?rios (libgl1, ffmpeg)...")
    subprocess.run(['sudo', 'apt-get', 'update', '-y'], check=False)
    subprocess.run(['sudo', 'apt-get', 'install', '-y', 'libgl1', 'ffmpeg'], check=False)
except Exception as e:
    print("Aviso ao tentar atualizar apt:", e)

print("Instalando depend?ncias do Python no ambiente MuseTalk...")
PYTHON_EXE = os.environ.get('PYTHON') or sys.executable
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)

# Vers?es compat?veis com o MuseTalk oficial e com wheels prebuilt do OpenMMLab.
subprocess.run([
    PYTHON_EXE, '-m', 'pip', 'install', '-q',
    'torch==2.0.1', 'torchvision==0.15.2', 'torchaudio==2.0.2',
    '--index-url', 'https://download.pytorch.org/whl/cu118'
], check=True)

req_file = REPO_DIR / 'requirements.txt'
if req_file.exists():
    content = req_file.read_text(encoding='utf-8')
    new_lines = []
    skip_tokens = ('torch', 'torchvision', 'torchaudio', 'tensorflow', 'tensorboard')
    for line in content.splitlines():
        line_clean = line.strip().lower()
        package_name = line_clean.split('==')[0].split('>=')[0].split('<=')[0].split('~=')[0]
        if package_name in skip_tokens:
            print(f'Comentando depend?ncia gerenciada manualmente: {line.strip()}')
            line = f'# {line}'
        elif line_clean.startswith('numpy'):
            line = 'numpy==1.26.4'
        elif line_clean.startswith('protobuf'):
            line = 'protobuf>=3.20.0'
        new_lines.append(line)
    req_file.write_text('\n'.join(new_lines), encoding='utf-8')
    subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-r', str(req_file)], check=True)

print("Instalando OpenMMLab com vers?es compat?veis...")
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', 'openmim', 'Cython', 'numpy==1.23.5'], check=True)
# mmpose puxa chumpy/xtcocotools. Pr?-instalamos para impedir que o resolver tente builds isolados quebrados.
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'chumpy==0.70'], check=True)
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'xtcocotools>=1.12'], check=True)
subprocess.run([PYTHON_EXE, '-m', 'mim', 'install', 'mmcv==2.0.1'], check=True)
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', 'mmengine', 'mmdet==3.1.0', 'mmpose==1.1.0'], check=True)

# Instala depend?ncias do microservi?o FastAPI
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn', 'python-multipart', 'huggingface_hub>=0.19.3,<1.0'], check=True)

# Smoke test cedo para n?o descobrir problema s? ao subir o servi?o.
subprocess.run([PYTHON_EXE, '-c', 'import torch, mmcv, mmengine, mmdet, mmpose; print("OK deps", torch.__version__, torch.version.cuda, mmcv.__version__)'], check=True)
print("Depend?ncias configuradas com sucesso!")


In [ ]:
# 3. Download de Pesos Oficiais do MuseTalk 1.5
import subprocess, sys, os
from pathlib import Path

PYTHON_EXE = os.environ.get('PYTHON') or sys.executable
print("Garantindo huggingface_hub e gdown compat?veis...")
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface_hub>=0.19.3,<1.0', 'gdown'], check=True)

models_dir = REPO_DIR / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

from huggingface_hub import snapshot_download, hf_hub_download

def hf_snapshot(repo_id, local_dir, allow_patterns):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    print(f'Baixando {repo_id} -> {local_dir}')
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(local_dir),
        allow_patterns=allow_patterns,
        local_dir_use_symlinks=False
    )

try:
    hf_snapshot('TMElyralab/MuseTalk', models_dir, [
        'musetalk/musetalk.json',
        'musetalk/pytorch_model.bin',
        'musetalkV15/musetalk.json',
        'musetalkV15/unet.pth'
    ])
    hf_snapshot('stabilityai/sd-vae-ft-mse', models_dir / 'sd-vae', [
        'config.json',
        'diffusion_pytorch_model.bin'
    ])
    hf_snapshot('openai/whisper-tiny', models_dir / 'whisper', [
        'config.json',
        'pytorch_model.bin',
        'preprocessor_config.json'
    ])
    hf_snapshot('yzd-v/DWPose', models_dir / 'dwpose', [
        'dw-ll_ucoco_384.pth'
    ])
    hf_snapshot('ByteDance/LatentSync', models_dir / 'syncnet', [
        'latentsync_syncnet.pt'
    ])

    face_dir = models_dir / 'face-parse-bisent'
    face_dir.mkdir(parents=True, exist_ok=True)

    def file_ok(path, min_bytes=1024 * 1024):
        return Path(path).exists() and Path(path).stat().st_size >= min_bytes

    if not file_ok(face_dir / '79999_iter.pth', 40 * 1024 * 1024):
        try:
            print('Baixando face-parse-bisent/79999_iter.pth via Hugging Face...')
            hf_hub_download(
                repo_id='ManyOtherFunctions/face-parse-bisent',
                filename='79999_iter.pth',
                local_dir=str(face_dir),
                local_dir_use_symlinks=False
            )
        except Exception as hf_face_err:
            print('Fallback Google Drive para 79999_iter.pth:', hf_face_err)
            subprocess.run([PYTHON_EXE, '-m', 'gdown', '--id', '154JgKpzCPW82qINcVieuPH3fZ2e0P812', '-O', str(face_dir / '79999_iter.pth')], check=True)

    if not file_ok(face_dir / 'resnet18-5c106cde.pth', 40 * 1024 * 1024):
        try:
            print('Baixando face-parse-bisent/resnet18-5c106cde.pth via Hugging Face...')
            hf_hub_download(
                repo_id='weege007/musetalk',
                filename='face-parse-bisent/resnet18-5c106cde.pth',
                local_dir=str(models_dir),
                local_dir_use_symlinks=False
            )
        except Exception as hf_resnet_err:
            print('Fallback download.pytorch.org para resnet18:', hf_resnet_err)
            subprocess.run(['curl', '-L', 'https://download.pytorch.org/models/resnet18-5c106cde.pth', '-o', str(face_dir / 'resnet18-5c106cde.pth')], check=True)

    print("Pesos baixados e organizados com sucesso!")
except Exception as e:
    print("Erro ao baixar pesos:", e)
    raise


In [ ]:
# 4. Valida??o da Estrutura dos Pesos
from pathlib import Path

models_dir = REPO_DIR / 'models'
expected_paths = [
    models_dir / 'musetalkV15' / 'unet.pth',
    models_dir / 'musetalkV15' / 'musetalk.json',
    models_dir / 'sd-vae' / 'config.json',
    models_dir / 'sd-vae' / 'diffusion_pytorch_model.bin',
    models_dir / 'whisper' / 'config.json',
    models_dir / 'whisper' / 'pytorch_model.bin',
    models_dir / 'whisper' / 'preprocessor_config.json',
    models_dir / 'dwpose' / 'dw-ll_ucoco_384.pth',
    models_dir / 'syncnet' / 'latentsync_syncnet.pt',
    models_dir / 'face-parse-bisent' / '79999_iter.pth',
    models_dir / 'face-parse-bisent' / 'resnet18-5c106cde.pth',
]

print('Estrutura de pesos local:')
missing = []
for p in expected_paths:
    ok = p.exists() and (p.is_dir() or p.stat().st_size > 0)
    status_str = '[OK]' if ok else '[FALHA]'
    print(f' {status_str} {p.relative_to(models_dir)}')
    if not ok:
        missing.append(str(p.relative_to(models_dir)))

if missing:
    raise RuntimeError('Pesos ausentes: ' + ', '.join(missing))
print('Todos os pesos necess?rios foram encontrados.')


In [ ]:
# 5. Escrever Arquivos do Microserviço
import base64
from pathlib import Path

app_b64 = "77u/ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbG9nZ2luZw0KaW1wb3J0IGFzeW5jaW8NCmltcG9ydCBqc29uDQppbXBvcnQgb3MNCmltcG9ydCByZQ0KaW1wb3J0IHNodXRpbA0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQpmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwNCg0KZnJvbSBmYXN0YXBpIGltcG9ydCBEZXBlbmRzLCBGYXN0QVBJLCBGaWxlLCBGb3JtLCBIZWFkZXIsIEhUVFBFeGNlcHRpb24sIFJlcXVlc3QsIFVwbG9hZEZpbGUsIHN0YXR1cw0KZnJvbSBmYXN0YXBpLnJlc3BvbnNlcyBpbXBvcnQgRmlsZVJlc3BvbnNlLCBTdHJlYW1pbmdSZXNwb25zZQ0KZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsLCBDb25maWdEaWN0LCBGaWVsZA0KDQpvcy5lbnZpcm9uWyJNUExCQUNLRU5EIl0gPSAiQWdnIg0KDQpmcm9tIG11c2V0YWxrX3NlcnZpY2UgaW1wb3J0ICgNCiAgICBHUFVVbmF2YWlsYWJsZUVycm9yLA0KICAgIE11c2VUYWxrRXhlY3V0aW9uRXJyb3IsDQogICAgTXVzZVRhbGtTZXJ2aWNlLA0KICAgIE11c2VUYWxrVGltZW91dEVycm9yLA0KKQ0KDQpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPW9zLmdldGVudigiTE9HX0xFVkVMIiwgIklORk8iKSwgZm9ybWF0PSIlKGFzY3RpbWUpcyAlKGxldmVsbmFtZSlzIFtMSVBTWU5DXSAlKG1lc3NhZ2UpcyIpDQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigibXJjaGlja2VuLmxpcHN5bmMiKQ0KDQphcHAgPSBGYXN0QVBJKHRpdGxlPSJNckNoaWNrZW4gTXVzZVRhbGsgTGlwLVN5bmMgU2VydmljZSIsIHZlcnNpb249IjEuMS4wIikNCnNlcnZpY2UgPSBNdXNlVGFsa1NlcnZpY2UoDQogICAgbW9kZWxzX2Rpcj1QYXRoKG9zLmdldGVudigiTVVTRVRBTEtfTU9ERUxTX0RJUiIsIFBhdGgoX19maWxlX18pLnBhcmVudCAvICJtb2RlbHMiKSksDQogICAgb3V0cHV0c19kaXI9UGF0aChvcy5nZXRlbnYoIk1VU0VUQUxLX09VVFBVVFNfRElSIiwgUGF0aChfX2ZpbGVfXykucGFyZW50IC8gIm91dHB1dHMiKSksDQopDQoNCg0KY2xhc3MgR2VuZXJhdGVSZXF1ZXN0KEJhc2VNb2RlbCk6DQogICAgam9iX2lkOiBzdHIgPSBGaWVsZCguLi4sIGFsaWFzPSJqb2JJZCIsIG1pbl9sZW5ndGg9MSkNCiAgICBhdmF0YXJfcGF0aDogc3RyID0gRmllbGQoLi4uLCBhbGlhcz0iYXZhdGFyUGF0aCIsIG1pbl9sZW5ndGg9MSkNCiAgICBhdWRpb19wYXRoOiBzdHIgPSBGaWVsZCguLi4sIGFsaWFzPSJhdWRpb1BhdGgiLCBtaW5fbGVuZ3RoPTEpDQoNCg0KY2xhc3MgR2VuZXJhdGVSZXNwb25zZShCYXNlTW9kZWwpOg0KICAgIG1vZGVsX2NvbmZpZyA9IENvbmZpZ0RpY3QocG9wdWxhdGVfYnlfbmFtZT1UcnVlKQ0KDQogICAgc3VjY2VzczogYm9vbA0KICAgIHByb3ZpZGVyOiBzdHIgPSAibXVzZXRhbGstdjE1Ig0KICAgIHZpZGVvX3BhdGg6IHN0ciA9IEZpZWxkKC4uLiwgYWxpYXM9InZpZGVvUGF0aCIpDQogICAgdmlkZW9fdXJsOiBPcHRpb25hbFtzdHJdID0gRmllbGQoZGVmYXVsdD1Ob25lLCBhbGlhcz0idmlkZW9VcmwiKQ0KDQoNCmNsYXNzIEVycm9yUmVzcG9uc2UoQmFzZU1vZGVsKToNCiAgICBzdWNjZXNzOiBib29sID0gRmFsc2UNCiAgICBwcm92aWRlcjogc3RyID0gIm11c2V0YWxrLXYxNSINCiAgICBlcnJvcjogc3RyDQogICAgY29kZTogc3RyDQogICAgZGV0YWlsczogT3B0aW9uYWxbc3RyXSA9IE5vbmUNCg0KDQpkZWYgdmVyaWZ5X2FwaV9rZXkoDQogICAgYXV0aG9yaXphdGlvbjogT3B0aW9uYWxbc3RyXSA9IEhlYWRlcihkZWZhdWx0PU5vbmUpLA0KICAgIHhfYXBpX2tleTogT3B0aW9uYWxbc3RyXSA9IEhlYWRlcihkZWZhdWx0PU5vbmUpLA0KKSAtPiBOb25lOg0KICAgICMgQVBJIGtleSB2ZXJpZmljYXRpb24gbG9naWMNCiAgICBleHBlY3RlZF9rZXkgPSBvcy5nZXRlbnYoIkxJUFNZTkNfQVBJX0tFWSIpDQogICAgaWYgbm90IGV4cGVjdGVkX2tleToNCiAgICAgICAgcmV0dXJuDQogICAgICAgIA0KICAgIHByb3ZpZGVkX2tleSA9IE5vbmUNCiAgICBpZiB4X2FwaV9rZXk6DQogICAgICAgIHByb3ZpZGVkX2tleSA9IHhfYXBpX2tleS5zdHJpcCgpDQogICAgZWxpZiBhdXRob3JpemF0aW9uIGFuZCBhdXRob3JpemF0aW9uLnN0YXJ0c3dpdGgoIkJlYXJlciAiKToNCiAgICAgICAgcHJvdmlkZWRfa2V5ID0gYXV0aG9yaXphdGlvbltsZW4oIkJlYXJlciAiKTpdLnN0cmlwKCkNCiAgICAgICAgDQogICAgaWYgcHJvdmlkZWRfa2V5ICE9IGV4cGVjdGVkX2tleS5zdHJpcCgpOg0KICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKA0KICAgICAgICAgICAgc3RhdHVzX2NvZGU9c3RhdHVzLkhUVFBfNDAxX1VOQVVUSE9SSVpFRCwNCiAgICAgICAgICAgIGRldGFpbD17DQogICAgICAgICAgICAgICAgInN1Y2Nlc3MiOiBGYWxzZSwNCiAgICAgICAgICAgICAgICAicHJvdmlkZXIiOiAibXVzZXRhbGstdjE1IiwNCiAgICAgICAgICAgICAgICAiZXJyb3IiOiAiQ2hhdmUgZGUgQVBJIGludsODwqFsaWRhIG91IGF1c2VudGUuIiwNCiAgICAgICAgICAgICAgICAiY29kZSI6ICJVTkFVVEhPUklaRUQiDQogICAgICAgICAgICB9DQogICAgICAgICkNCg0KDQpkZWYgc2FmZV9wYXRoX3NlZ21lbnQodmFsdWU6IHN0cikgLT4gc3RyOg0KICAgIHJldHVybiByZS5zdWIociJbXmEtekEtWjAtOS5fLV0iLCAiLSIsIHZhbHVlKVs6MTIwXSBvciAiam9iIg0KDQoNCmRlZiBzYWZlX3VwbG9hZF9uYW1lKGZpbGVuYW1lOiBzdHIgfCBOb25lLCBmYWxsYmFjazogc3RyKSAtPiBzdHI6DQogICAgc3VmZml4ID0gUGF0aChmaWxlbmFtZSBvciBmYWxsYmFjaykuc3VmZml4Lmxvd2VyKCkNCiAgICBpZiBzdWZmaXggbm90IGluIHsiLmpwZyIsICIuanBlZyIsICIucG5nIiwgIi53ZWJwIiwgIi5tcDQiLCAiLm1vdiIsICIud2VibSIsICIud2F2IiwgIi5tcDMiLCAiLm00YSJ9Og0KICAgICAgICBzdWZmaXggPSBQYXRoKGZhbGxiYWNrKS5zdWZmaXgNCiAgICByZXR1cm4gZiJ7UGF0aChmYWxsYmFjaykuc3RlbX17c3VmZml4fSINCg0KDQpkZWYgc2F2ZV91cGxvYWQodXBsb2FkOiBVcGxvYWRGaWxlLCBkZXN0aW5hdGlvbjogUGF0aCkgLT4gUGF0aDoNCiAgICBkZXN0aW5hdGlvbi5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIHdpdGggZGVzdGluYXRpb24ub3Blbigid2IiKSBhcyBvdXRwdXQ6DQogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaih1cGxvYWQuZmlsZSwgb3V0cHV0KQ0KICAgIHJldHVybiBkZXN0aW5hdGlvbg0KDQoNCmRlZiBlcnJvcl9yZXNwb25zZShzdGF0dXNfY29kZTogaW50LCBlcnJvcjogc3RyLCBjb2RlOiBzdHIsIGRldGFpbHM6IE9wdGlvbmFsW3N0cl0gPSBOb25lKSAtPiBIVFRQRXhjZXB0aW9uOg0KICAgIHJldHVybiBIVFRQRXhjZXB0aW9uKA0KICAgICAgICBzdGF0dXNfY29kZT1zdGF0dXNfY29kZSwNCiAgICAgICAgZGV0YWlsPXsNCiAgICAgICAgICAgICJzdWNjZXNzIjogRmFsc2UsDQogICAgICAgICAgICAicHJvdmlkZXIiOiAibXVzZXRhbGstdjE1IiwNCiAgICAgICAgICAgICJlcnJvciI6IGVycm9yLA0KICAgICAgICAgICAgImNvZGUiOiBjb2RlLA0KICAgICAgICAgICAgImRldGFpbHMiOiBkZXRhaWxzIG9yICIiDQogICAgICAgIH0sDQogICAgKQ0KDQoNCmRlZiBydW5fZ2VuZXJhdGlvbihqb2JfaWQ6IHN0ciwgYXZhdGFyX3BhdGg6IFBhdGgsIGF1ZGlvX3BhdGg6IFBhdGgpIC0+IFBhdGg6DQogICAgdHJ5Og0KICAgICAgICByZXR1cm4gc2VydmljZS5nZW5lcmF0ZSgNCiAgICAgICAgICAgIGpvYl9pZD1qb2JfaWQsDQogICAgICAgICAgICBhdmF0YXJfcGF0aD1hdmF0YXJfcGF0aCwNCiAgICAgICAgICAgIGF1ZGlvX3BhdGg9YXVkaW9fcGF0aCwNCiAgICAgICAgKQ0KICAgIGV4Y2VwdCBGaWxlTm90Rm91bmRFcnJvciBhcyBleGM6DQogICAgICAgIHJhaXNlIGVycm9yX3Jlc3BvbnNlKHN0YXR1cy5IVFRQXzQwNF9OT1RfRk9VTkQsIHN0cihleGMpLCAiRklMRV9OT1RfRk9VTkQiKSBmcm9tIGV4Yw0KICAgIGV4Y2VwdCBHUFVVbmF2YWlsYWJsZUVycm9yIGFzIGV4YzoNCiAgICAgICAgcmFpc2UgZXJyb3JfcmVzcG9uc2Uoc3RhdHVzLkhUVFBfNTAzX1NFUlZJQ0VfVU5BVkFJTEFCTEUsIHN0cihleGMpLCAiR1BVX1VOQVZBSUxBQkxFIikgZnJvbSBleGMNCiAgICBleGNlcHQgTXVzZVRhbGtUaW1lb3V0RXJyb3IgYXMgZXhjOg0KICAgICAgICByYWlzZSBlcnJvcl9yZXNwb25zZShzdGF0dXMuSFRUUF81MDRfR0FURVdBWV9USU1FT1VULCBzdHIoZXhjKSwgIlRJTUVPVVQiKSBmcm9tIGV4Yw0KICAgIGV4Y2VwdCBNdXNlVGFsa0V4ZWN1dGlvbkVycm9yIGFzIGV4YzoNCiAgICAgICAgam9iX2RpciA9IHNlcnZpY2Uub3V0cHV0c19kaXIgLyBqb2JfaWQNCiAgICAgICAgZXJyb3JfZGV0YWlscyA9ICIiDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGVycl9sb2dfcGF0aCA9IGpvYl9kaXIgLyAiZXJyb3IubG9nIg0KICAgICAgICAgICAgaWYgZXJyX2xvZ19wYXRoLmV4aXN0cygpOg0KICAgICAgICAgICAgICAgIGVycm9yX2RldGFpbHMgPSBlcnJfbG9nX3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpWy0xMDAwOl0NCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgc3RkZXJyX2xvZ19wYXRoID0gam9iX2RpciAvICJzdGRlcnIubG9nIg0KICAgICAgICAgICAgICAgIGlmIHN0ZGVycl9sb2dfcGF0aC5leGlzdHMoKToNCiAgICAgICAgICAgICAgICAgICAgZXJyb3JfZGV0YWlscyA9IHN0ZGVycl9sb2dfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IilbLTEwMDA6XQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICAgICAgcGFzcw0KICAgICAgICByYWlzZSBlcnJvcl9yZXNwb25zZSgNCiAgICAgICAgICAgIHN0YXR1c19jb2RlPXN0YXR1cy5IVFRQXzUwMF9JTlRFUk5BTF9TRVJWRVJfRVJST1IsDQogICAgICAgICAgICBlcnJvcj1zdHIoZXhjKSwNCiAgICAgICAgICAgIGNvZGU9Ik1VU0VUQUxLX0VSUk9SIiwNCiAgICAgICAgICAgIGRldGFpbHM9ZXJyb3JfZGV0YWlscyBvciBzdHIoZXhjKQ0KICAgICAgICApIGZyb20gZXhjDQoNCg0KQGFwcC5nZXQoIi9oZWFsdGgiKQ0KZGVmIGhlYWx0aCgpIC0+IGRpY3Rbc3RyLCBvYmplY3RdOg0KICAgIGltcG9ydHNfc3RhdHVzID0ge30NCiAgICBjcml0aWNhbF9saWJzID0gWyJ0b3JjaCIsICJkaWZmdXNlcnMiLCAidHJhbnNmb3JtZXJzIiwgIm1tcG9zZSIsICJtbWN2IiwgIm1tZGV0IiwgIm1tZW5naW5lIl0NCiAgICBmb3IgbGliIGluIGNyaXRpY2FsX2xpYnM6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIF9faW1wb3J0X18obGliKQ0KICAgICAgICAgICAgaW1wb3J0c19zdGF0dXNbbGliXSA9ICJPSyINCiAgICAgICAgZXhjZXB0IEltcG9ydEVycm9yIGFzIGU6DQogICAgICAgICAgICBpbXBvcnRzX3N0YXR1c1tsaWJdID0gZiJNaXNzaW5nOiB7c3RyKGUpfSINCiAgICAgICAgICAgIA0KICAgIGdwdV9hdmFpbCA9IHNlcnZpY2UuZ3B1X2F2YWlsYWJsZSgpDQogICAgDQogICAgY29uZmlnX2RldGFpbHMgPSB7DQogICAgICAgICJtdXNldGFsa192ZXJzaW9uIjogb3MuZ2V0ZW52KCJNVVNFVEFMS19WRVJTSU9OIiwgInYxNSIpLA0KICAgICAgICAidW5ldF9tb2RlbF9wYXRoIjogb3MuZ2V0ZW52KCJNVVNFVEFMS19VTkVUX01PREVMX1BBVEgiLCAibW9kZWxzL211c2V0YWxrVjE1L3VuZXQucHRoIiksDQogICAgICAgICJ1bmV0X2NvbmZpZyI6IG9zLmdldGVudigiTVVTRVRBTEtfVU5FVF9DT05GSUciLCAibW9kZWxzL211c2V0YWxrVjE1L211c2V0YWxrLmpzb24iKSwNCiAgICAgICAgInJlcXVpcmVfZ3B1Ijogb3MuZ2V0ZW52KCJNVVNFVEFMS19SRVFVSVJFX0dQVSIsICJ0cnVlIiksDQogICAgICAgICJ0aW1lb3V0X3NlY29uZHMiOiBvcy5nZXRlbnYoIk1VU0VUQUxLX1RJTUVPVVRfU0VDT05EUyIsICIxODAwIiksDQogICAgfQ0KICAgIA0KICAgIHJldHVybiB7DQogICAgICAgICJzdWNjZXNzIjogVHJ1ZSwNCiAgICAgICAgImVuZ2luZSI6ICJtdXNldGFsay12MTUiLA0KICAgICAgICAiZ3B1QXZhaWxhYmxlIjogZ3B1X2F2YWlsLA0KICAgICAgICAicmVwb1BhdGgiOiBvcy5nZXRlbnYoIk1VU0VUQUxLX1JFUE9fUEFUSCIsICIiKSwNCiAgICAgICAgIm91dHB1dHNQYXRoIjogc3RyKHNlcnZpY2Uub3V0cHV0c19kaXIucmVzb2x2ZSgpKSwNCiAgICAgICAgImNvbmZpZyI6IGNvbmZpZ19kZXRhaWxzLA0KICAgICAgICAiaW1wb3J0cyI6IGltcG9ydHNfc3RhdHVzDQogICAgfQ0KDQoNCkBhcHAucG9zdCgiL2dlbmVyYXRlIiwgcmVzcG9uc2VfbW9kZWw9R2VuZXJhdGVSZXNwb25zZSwgcmVzcG9uc2VzPXsNCiAgICA0MDE6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwNCiAgICA0MDQ6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwNCiAgICA1MDA6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwNCiAgICA1MDM6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwNCiAgICA1MDQ6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwNCn0pDQpkZWYgZ2VuZXJhdGUocmVxdWVzdDogR2VuZXJhdGVSZXF1ZXN0LCBfOiBOb25lID0gRGVwZW5kcyh2ZXJpZnlfYXBpX2tleSkpIC0+IEdlbmVyYXRlUmVzcG9uc2U6DQogICAgbG9nZ2VyLmluZm8oIkpvYiAlcyByZWNlYmlkbzogYXZhdGFyPSVzIGF1ZGlvPSVzIiwgcmVxdWVzdC5qb2JfaWQsIHJlcXVlc3QuYXZhdGFyX3BhdGgsIHJlcXVlc3QuYXVkaW9fcGF0aCkNCiAgICBhc3luYyBkZWYgZXZlbnRfZ2VuZXJhdG9yKCk6DQogICAgICAgIGxvb3AgPSBhc3luY2lvLmdldF9ydW5uaW5nX2xvb3AoKQ0KICAgICAgICB0YXNrID0gbG9vcC5ydW5faW5fZXhlY3V0b3IoTm9uZSwgcnVuX2dlbmVyYXRpb24sIHNhZmVfam9iX2lkLCBhdmF0YXJfcGF0aCwgYXVkaW9fcGF0aCkNCg0KICAgICAgICB3aGlsZSBub3QgdGFzay5kb25lKCk6DQogICAgICAgICAgICB5aWVsZCAiICINCiAgICAgICAgICAgIGF3YWl0IGFzeW5jaW8uc2xlZXAoNSkNCg0KICAgICAgICB0cnk6DQogICAgICAgICAgICB2aWRlb19wYXRoID0gYXdhaXQgdGFzaw0KICAgICAgICAgICAgdmlkZW9fdXJsID0gc3RyKHJlcXVlc3QudXJsX2ZvcigiZG93bmxvYWRfb3V0cHV0Iiwgam9iX2lkPXNhZmVfam9iX2lkLCBmaWxlbmFtZT12aWRlb19wYXRoLm5hbWUpKQ0KICAgICAgICAgICAgbG9nZ2VyLmluZm8oIkpvYiAlcyBjb25jbHXDrWRvIHZpYSB1cGxvYWQ6ICVzIiwgam9iSWQsIHZpZGVvX3BhdGgpDQogICAgICAgICAgICB5aWVsZCBqc29uLmR1bXBzKHsNCiAgICAgICAgICAgICAgICAic3VjY2VzcyI6IFRydWUsDQogICAgICAgICAgICAgICAgInByb3ZpZGVyIjogIm11c2V0YWxrLXYxNSIsDQogICAgICAgICAgICAgICAgInZpZGVvUGF0aCI6IHN0cih2aWRlb19wYXRoKSwNCiAgICAgICAgICAgICAgICAidmlkZW9VcmwiOiB2aWRlb191cmwsDQogICAgICAgICAgICB9KQ0KICAgICAgICBleGNlcHQgSFRUUEV4Y2VwdGlvbiBhcyBleGM6DQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKGV4Yy5kZXRhaWwsIGRpY3QpOg0KICAgICAgICAgICAgICAgIHlpZWxkIGpzb24uZHVtcHMoZXhjLmRldGFpbCkNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgeWllbGQganNvbi5kdW1wcyh7DQogICAgICAgICAgICAgICAgICAgICJzdWNjZXNzIjogRmFsc2UsDQogICAgICAgICAgICAgICAgICAgICJwcm92aWRlciI6ICJtdXNldGFsay12MTUiLA0KICAgICAgICAgICAgICAgICAgICAiZXJyb3IiOiBzdHIoZXhjLmRldGFpbCksDQogICAgICAgICAgICAgICAgICAgICJjb2RlIjogIk1VU0VUQUxLX0VSUk9SIiwNCiAgICAgICAgICAgICAgICB9KQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoNCiAgICAgICAgICAgIHlpZWxkIGpzb24uZHVtcHMoew0KICAgICAgICAgICAgICAgICJzdWNjZXNzIjogRmFsc2UsDQogICAgICAgICAgICAgICAgInByb3ZpZGVyIjogIm11c2V0YWxrLXYxNSIsDQogICAgICAgICAgICAgICAgImVycm9yIjogc3RyKGV4YyksDQogICAgICAgICAgICAgICAgImNvZGUiOiAiTVVTRVRBTEtfRVJST1IiLA0KICAgICAgICAgICAgfSkNCg0KICAgIHJldHVybiBTdHJlYW1pbmdSZXNwb25zZShldmVudF9nZW5lcmF0b3IoKSwgbWVkaWFfdHlwZT0iYXBwbGljYXRpb24vanNvbiIpDQoNCg0KQGFwcC5wb3N0KCIvZ2VuZXJhdGUtdXBsb2FkIiwgcmVzcG9uc2VzPXsNCiAgICA0MDE6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwNCiAgICA0MDQ6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwNCiAgICA1MDA6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwNCiAgICA1MDM6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwNCiAgICA1MDQ6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwNCn0pDQphc3luYyBkZWYgZ2VuZXJhdGVfdXBsb2FkKA0KICAgIHJlcXVlc3Q6IFJlcXVlc3QsDQogICAgam9iSWQ6IHN0ciA9IEZvcm0oLi4uLCBtaW5fbGVuZ3RoPTEpLA0KICAgIGF2YXRhcjogT3B0aW9uYWxbVXBsb2FkRmlsZV0gPSBGaWxlKGRlZmF1bHQ9Tm9uZSksDQogICAgdmlkZW86IE9wdGlvbmFsW1VwbG9hZEZpbGVdID0gRmlsZShkZWZhdWx0PU5vbmUpLA0KICAgIGF1ZGlvOiBVcGxvYWRGaWxlID0gRmlsZSguLi4pLA0KICAgIF86IE5vbmUgPSBEZXBlbmRzKHZlcmlmeV9hcGlfa2V5KSwNCikgLT4gU3RyZWFtaW5nUmVzcG9uc2U6DQogICAgc2FmZV9qb2JfaWQgPSBzYWZlX3BhdGhfc2VnbWVudChqb2JJZCkNCiAgICBqb2JfZGlyID0gc2VydmljZS5vdXRwdXRzX2RpciAvIHNhZmVfam9iX2lkDQogICAgDQogICAgdXBsb2FkX2ZpbGUgPSBhdmF0YXIgb3IgdmlkZW8NCiAgICBpZiBub3QgdXBsb2FkX2ZpbGU6DQogICAgICAgIHJhaXNlIGVycm9yX3Jlc3BvbnNlKA0KICAgICAgICAgICAgc3RhdHVzX2NvZGU9c3RhdHVzLkhUVFBfNDAwX0JBRF9SRVFVRVNULA0KICAgICAgICAgICAgZXJyb3I9IkF2YXRhciBvdSB2aWRlbyDDg8KpIG9icmlnYXTDg8KzcmlvLiIsDQogICAgICAgICAgICBjb2RlPSJGSUxFX05PVF9GT1VORCINCiAgICAgICAgKQ0KICAgICAgICANCiAgICBhdmF0YXJfcGF0aCA9IHNhdmVfdXBsb2FkKHVwbG9hZF9maWxlLCBqb2JfZGlyIC8gc2FmZV91cGxvYWRfbmFtZSh1cGxvYWRfZmlsZS5maWxlbmFtZSwgImF2YXRhci5qcGciKSkNCiAgICBhdWRpb19wYXRoID0gc2F2ZV91cGxvYWQoYXVkaW8sIGpvYl9kaXIgLyBzYWZlX3VwbG9hZF9uYW1lKGF1ZGlvLmZpbGVuYW1lLCAiYXVkaW8ud2F2IikpDQogICAgbG9nZ2VyLmluZm8oIkpvYiAlcyByZWNlYmlkbyB2aWEgdXBsb2FkOiBhdmF0YXIvdmlkZW89JXMgYXVkaW89JXMiLCBqb2JJZCwgYXZhdGFyX3BhdGgsIGF1ZGlvX3BhdGgpDQoNCiAgICB2aWRlb19wYXRoID0gcnVuX2dlbmVyYXRpb24oDQogICAgICAgIGpvYl9pZD1zYWZlX2pvYl9pZCwNCiAgICAgICAgYXZhdGFyX3BhdGg9YXZhdGFyX3BhdGgsDQogICAgICAgIGF1ZGlvX3BhdGg9YXVkaW9fcGF0aCwNCiAgICApDQogICAgdmlkZW9fdXJsID0gc3RyKHJlcXVlc3QudXJsX2ZvcigiZG93bmxvYWRfb3V0cHV0Iiwgam9iX2lkPXNhZmVfam9iX2lkLCBmaWxlbmFtZT12aWRlb19wYXRoLm5hbWUpKQ0KICAgIGxvZ2dlci5pbmZvKCJKb2IgJXMgY29uY2x1w4PCrWRvIHZpYSB1cGxvYWQ6ICVzIiwgam9iSWQsIHZpZGVvX3BhdGgpDQogICAgcmV0dXJuIEdlbmVyYXRlUmVzcG9uc2Uoc3VjY2Vzcz1UcnVlLCBwcm92aWRlcj0ibXVzZXRhbGstdjE1IiwgdmlkZW9fcGF0aD1zdHIodmlkZW9fcGF0aCksIHZpZGVvX3VybD12aWRlb191cmwpDQoNCg0KQGFwcC5nZXQoIi9vdXRwdXRzL3tqb2JfaWR9L3tmaWxlbmFtZX0iLCBuYW1lPSJkb3dubG9hZF9vdXRwdXQiKQ0KZGVmIGRvd25sb2FkX291dHB1dChqb2JfaWQ6IHN0ciwgZmlsZW5hbWU6IHN0ciwgXzogTm9uZSA9IERlcGVuZHModmVyaWZ5X2FwaV9rZXkpKSAtPiBGaWxlUmVzcG9uc2U6DQogICAgc2FmZV9qb2JfaWQgPSBzYWZlX3BhdGhfc2VnbWVudChqb2JfaWQpDQogICAgc2FmZV9maWxlbmFtZSA9IFBhdGgoZmlsZW5hbWUpLm5hbWUNCiAgICBvdXRwdXRfcGF0aCA9IHNlcnZpY2Uub3V0cHV0c19kaXIgLyBzYWZlX2pvYl9pZCAvIHNhZmVfZmlsZW5hbWUNCiAgICBpZiBub3Qgb3V0cHV0X3BhdGguZXhpc3RzKCkgb3Igbm90IG91dHB1dF9wYXRoLmlzX2ZpbGUoKToNCiAgICAgICAgcmFpc2UgSFRUUEV4Y2VwdGlvbigNCiAgICAgICAgICAgIHN0YXR1c19jb2RlPXN0YXR1cy5IVFRQXzQwNF9OT1RfRk9VTkQsDQogICAgICAgICAgICBkZXRhaWw9ew0KICAgICAgICAgICAgICAgICJzdWNjZXNzIjogRmFsc2UsDQogICAgICAgICAgICAgICAgInByb3ZpZGVyIjogIm11c2V0YWxrLXYxNSIsDQogICAgICAgICAgICAgICAgImVycm9yIjogZiJPdXRwdXQgbsODwqNvIGVuY29udHJhZG86IHtzYWZlX2ZpbGVuYW1lfSIsDQogICAgICAgICAgICAgICAgImNvZGUiOiAiRklMRV9OT1RfRk9VTkQiDQogICAgICAgICAgICB9LA0KICAgICAgICApDQogICAgcmV0dXJuIEZpbGVSZXNwb25zZShvdXRwdXRfcGF0aCwgbWVkaWFfdHlwZT0idmlkZW8vbXA0IiwgZmlsZW5hbWU9c2FmZV9maWxlbmFtZSkNCg=="
service_b64 = "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGdsb2IKaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCBzaGxleAppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gc3RyaW5nIGltcG9ydCBUZW1wbGF0ZQoKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoIm1yY2hpY2tlbi5saXBzeW5jIikKCgpjbGFzcyBHUFVVbmF2YWlsYWJsZUVycm9yKFJ1bnRpbWVFcnJvcik6CiAgICAiIiJSYWlzZWQgd2hlbiBNdXNlVGFsayBpcyBjb25maWd1cmVkIHRvIHJlcXVpcmUgQ1VEQSBidXQgbm8gR1BVIGlzIHZpc2libGUuIiIiCgoKY2xhc3MgTXVzZVRhbGtFeGVjdXRpb25FcnJvcihSdW50aW1lRXJyb3IpOgogICAgIiIiUmFpc2VkIHdoZW4gdGhlIE11c2VUYWxrIHByb2Nlc3MgZXhpdHMgd2l0aCBhIG5vbi16ZXJvIHN0YXR1cyBvciBubyBvdXRwdXQuIiIiCgoKY2xhc3MgTXVzZVRhbGtUaW1lb3V0RXJyb3IoVGltZW91dEVycm9yKToKICAgICIiIlJhaXNlZCB3aGVuIE11c2VUYWxrIGV4Y2VlZHMgdGhlIGNvbmZpZ3VyZWQgdGltZW91dC4iIiIKCgpjbGFzcyBNdXNlVGFsa1NlcnZpY2U6CiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWxzX2RpcjogUGF0aCwgb3V0cHV0c19kaXI6IFBhdGgpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5tb2RlbHNfZGlyID0gbW9kZWxzX2RpcgogICAgICAgIHNlbGYub3V0cHV0c19kaXIgPSBvdXRwdXRzX2RpcgogICAgICAgIHNlbGYubW9kZWxzX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgc2VsZi5vdXRwdXRzX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgZGVmIGdwdV9hdmFpbGFibGUoc2VsZikgLT4gYm9vbDoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGltcG9ydCB0b3JjaCAgIyB0eXBlOiBpZ25vcmUKCiAgICAgICAgICAgIGlmIGJvb2wodG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSk6CiAgICAgICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgICAgIFsibnZpZGlhLXNtaSIsICItLXF1ZXJ5LWdwdT1uYW1lIiwgIi0tZm9ybWF0PWNzdixub2hlYWRlciJdLAogICAgICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwKICAgICAgICAgICAgICAgIHRleHQ9VHJ1ZSwKICAgICAgICAgICAgICAgIHRpbWVvdXQ9NSwKICAgICAgICAgICAgICAgIGNoZWNrPUZhbHNlLAogICAgICAgICAgICApCiAgICAgICAgICAgIHJldHVybiByZXN1bHQucmV0dXJuY29kZSA9PSAwIGFuZCBib29sKHJlc3VsdC5zdGRvdXQuc3RyaXAoKSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKCiAgICBkZWYgZ2VuZXJhdGUoc2VsZiwgam9iX2lkOiBzdHIsIGF2YXRhcl9wYXRoOiBQYXRoLCBhdWRpb19wYXRoOiBQYXRoKSAtPiBQYXRoOgogICAgICAgIHNlbGYuX3ZhbGlkYXRlX2lucHV0cyhhdmF0YXJfcGF0aD1hdmF0YXJfcGF0aCwgYXVkaW9fcGF0aD1hdWRpb19wYXRoKQogICAgICAgIHNlbGYuX3ZhbGlkYXRlX2dwdSgpCgogICAgICAgIGpvYl9kaXIgPSBzZWxmLm91dHB1dHNfZGlyIC8gam9iX2lkCiAgICAgICAgam9iX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgCiAgICAgICAgdmVyc2lvbiA9IG9zLmdldGVudigiTVVTRVRBTEtfVkVSU0lPTiIsICJ2MTUiKQogICAgICAgIGZpbGVuYW1lID0gZiJtdXNldGFsay17dmVyc2lvbn0tb3V0cHV0Lm1wNCIgaWYgdmVyc2lvbiBlbHNlICJtdXNldGFsay1vdXRwdXQubXA0IgogICAgICAgIG91dHB1dF9wYXRoID0gam9iX2RpciAvIGZpbGVuYW1lCgogICAgICAgIGNvbW1hbmQgPSBzZWxmLl9idWlsZF9jb21tYW5kKAogICAgICAgICAgICBqb2JfaWQ9am9iX2lkLAogICAgICAgICAgICBhdmF0YXJfcGF0aD1hdmF0YXJfcGF0aCwKICAgICAgICAgICAgYXVkaW9fcGF0aD1hdWRpb19wYXRoLAogICAgICAgICAgICBvdXRwdXRfcGF0aD1vdXRwdXRfcGF0aCwKICAgICAgICAgICAgam9iX2Rpcj1qb2JfZGlyLAogICAgICAgICkKICAgICAgICB0aW1lb3V0ID0gaW50KG9zLmdldGVudigiTVVTRVRBTEtfVElNRU9VVF9TRUNPTkRTIiwgIjE4MDAiKSkKICAgICAgICBjd2QgPSBvcy5nZXRlbnYoIk1VU0VUQUxLX1JFUE9fUEFUSCIpIG9yIE5vbmUKCiAgICAgICAgIyBCdWlsZCBjdXN0b20gZW52aXJvbm1lbnQgdG8gcGFzcyBGRk1QRUdfUEFUSCBhbmQgUFlUSE9OUEFUSAogICAgICAgIGVudiA9IG9zLmVudmlyb24uY29weSgpCiAgICAgICAgZW52WyJNUExCQUNLRU5EIl0gPSAiQWdnIgogICAgICAgIGZmbXBlZ19wYXRoID0gb3MuZ2V0ZW52KCJNVVNFVEFMS19GRk1QRUdfUEFUSCIpCiAgICAgICAgaWYgZmZtcGVnX3BhdGg6CiAgICAgICAgICAgIGVudlsiRkZNUEVHX1BBVEgiXSA9IGZmbXBlZ19wYXRoCiAgICAgICAgICAgICMgQWxzbyBwcmVwZW5kIHRvIHN5c3RlbSBwYXRoIGlmIG5lZWRlZAogICAgICAgICAgICBmZm1wZWdfZGlyID0gc3RyKFBhdGgoZmZtcGVnX3BhdGgpLnBhcmVudCkKICAgICAgICAgICAgZW52WyJQQVRIIl0gPSBmZm1wZWdfZGlyICsgb3MucGF0aHNlcCArIGVudi5nZXQoIlBBVEgiLCAiIikKCiAgICAgICAgaWYgY3dkOgogICAgICAgICAgICBlbnZbIlBZVEhPTlBBVEgiXSA9IGYie2N3ZH17b3MucGF0aHNlcH17ZW52LmdldCgnUFlUSE9OUEFUSCcsICcnKX0iCgogICAgICAgIHN0ZG91dF9maWxlID0gam9iX2RpciAvICJzdGRvdXQubG9nIgogICAgICAgIHN0ZGVycl9maWxlID0gam9iX2RpciAvICJzdGRlcnIubG9nIgogICAgICAgIGVycm9yX2ZpbGUgPSBqb2JfZGlyIC8gImVycm9yLmxvZyIKCiAgICAgICAgbG9nZ2VyLmluZm8oIkV4ZWN1dGFuZG8gTXVzZVRhbGsgcGFyYSBqb2IgJXM6ICVzIiwgam9iX2lkLCAiICIuam9pbihjb21tYW5kKSkKICAgICAgICBzdGFydGVkID0gdGltZS5tb25vdG9uaWMoKQogICAgICAgIHRyeToKICAgICAgICAgICAgcmVzdWx0ID0gc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgICAgICAgICBjb21tYW5kLAogICAgICAgICAgICAgICAgY3dkPWN3ZCwKICAgICAgICAgICAgICAgIGVudj1lbnYsCiAgICAgICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLAogICAgICAgICAgICAgICAgdGV4dD1UcnVlLAogICAgICAgICAgICAgICAgdGltZW91dD10aW1lb3V0LAogICAgICAgICAgICAgICAgY2hlY2s9RmFsc2UsCiAgICAgICAgICAgICkKICAgICAgICBleGNlcHQgc3VicHJvY2Vzcy5UaW1lb3V0RXhwaXJlZCBhcyBleGM6CiAgICAgICAgICAgIGVycm9yX2ZpbGUud3JpdGVfdGV4dChmIlRpbWVvdXRFeHBpcmVkOiBNdXNlVGFsayBleGNlZGV1IG8gdGltZW91dCBkZSB7dGltZW91dH1zLiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgICAgIHJhaXNlIE11c2VUYWxrVGltZW91dEVycm9yKGYiTXVzZVRhbGsgZXhjZWRldSBvIHRpbWVvdXQgZGUge3RpbWVvdXR9cyBwYXJhIG8gam9iIHtqb2JfaWR9LiIpIGZyb20gZXhjCgogICAgICAgIGVsYXBzZWQgPSB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZAogICAgICAgIAogICAgICAgICMgU2F2ZSBzdGRvdXQvc3RkZXJyIGxvZ3MKICAgICAgICBzdGRvdXRfZmlsZS53cml0ZV90ZXh0KHJlc3VsdC5zdGRvdXQgb3IgIiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgc3RkZXJyX2ZpbGUud3JpdGVfdGV4dChyZXN1bHQuc3RkZXJyIG9yICIiLCBlbmNvZGluZz0idXRmLTgiKQoKICAgICAgICBpZiByZXN1bHQuc3Rkb3V0OgogICAgICAgICAgICBsb2dnZXIuaW5mbygiTXVzZVRhbGsgc3Rkb3V0IGpvYiAlczpcbiVzIiwgam9iX2lkLCByZXN1bHQuc3Rkb3V0Wy0yMDAwOl0pCiAgICAgICAgaWYgcmVzdWx0LnN0ZGVycjoKICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoIk11c2VUYWxrIHN0ZGVyciBqb2IgJXM6XG4lcyIsIGpvYl9pZCwgcmVzdWx0LnN0ZGVyclstMjAwMDpdKQoKICAgICAgICBpZiByZXN1bHQucmV0dXJuY29kZSAhPSAwOgogICAgICAgICAgICB0cmFjZWJhY2tfc25pcHBldCA9IChyZXN1bHQuc3RkZXJyIG9yICIiKVstMjAwMDpdCiAgICAgICAgICAgIGVycm9yX21zZyA9IGYiTXVzZVRhbGsgZmFsaG91IGNvbSBjw7NkaWdvIHtyZXN1bHQucmV0dXJuY29kZX06XG57dHJhY2ViYWNrX3NuaXBwZXR9IgogICAgICAgICAgICBlcnJvcl9maWxlLndyaXRlX3RleHQoZiJSZXR1cm4gY29kZToge3Jlc3VsdC5yZXR1cm5jb2RlfVxuXG5TdGRlcnI6XG57cmVzdWx0LnN0ZGVycn1cblxuU3Rkb3V0Olxue3Jlc3VsdC5zdGRvdXR9IiwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICAgICAgcmFpc2UgTXVzZVRhbGtFeGVjdXRpb25FcnJvcihlcnJvcl9tc2cpCgogICAgICAgIGZpbmFsX3BhdGggPSBzZWxmLl9yZXNvbHZlX291dHB1dChqb2JfZGlyPWpvYl9kaXIsIGV4cGVjdGVkX3BhdGg9b3V0cHV0X3BhdGgpCiAgICAgICAgbG9nZ2VyLmluZm8oIk11c2VUYWxrIGZpbmFsaXpvdSBqb2IgJXMgZW0gJS4xZnM6ICVzIiwgam9iX2lkLCBlbGFwc2VkLCBmaW5hbF9wYXRoKQogICAgICAgIHJldHVybiBmaW5hbF9wYXRoCgogICAgZGVmIF92YWxpZGF0ZV9pbnB1dHMoc2VsZiwgYXZhdGFyX3BhdGg6IFBhdGgsIGF1ZGlvX3BhdGg6IFBhdGgpIC0+IE5vbmU6CiAgICAgICAgaWYgbm90IGF2YXRhcl9wYXRoLmV4aXN0cygpOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkFycXVpdm8gZGUgYXZhdGFyIG7Do28gZW5jb250cmFkbzoge2F2YXRhcl9wYXRofSIpCiAgICAgICAgaWYgbm90IGF1ZGlvX3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiQXJxdWl2byBkZSDDoXVkaW8gbsOjbyBlbmNvbnRyYWRvOiB7YXVkaW9fcGF0aH0iKQoKICAgIGRlZiBfdmFsaWRhdGVfZ3B1KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcmVxdWlyZV9ncHUgPSBvcy5nZXRlbnYoIk1VU0VUQUxLX1JFUVVJUkVfR1BVIiwgInRydWUiKS5sb3dlcigpIGluIHsiMSIsICJ0cnVlIiwgInllcyIsICJzaW0ifQogICAgICAgIGlmIHJlcXVpcmVfZ3B1IGFuZCBub3Qgc2VsZi5ncHVfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgIHJhaXNlIEdQVVVuYXZhaWxhYmxlRXJyb3IoIkdQVS9DVURBIGluZGlzcG9uw612ZWwgcGFyYSBleGVjdXRhciBNdXNlVGFsay4iKQoKICAgIGRlZiBfYnVpbGRfY29tbWFuZChzZWxmLCBqb2JfaWQ6IHN0ciwgYXZhdGFyX3BhdGg6IFBhdGgsIGF1ZGlvX3BhdGg6IFBhdGgsIG91dHB1dF9wYXRoOiBQYXRoLCBqb2JfZGlyOiBQYXRoKSAtPiBsaXN0W3N0cl06CiAgICAgICAgdGVtcGxhdGUgPSBvcy5nZXRlbnYoIk1VU0VUQUxLX0NPTU1BTkRfVEVNUExBVEUiKQogICAgICAgIGlmIHRlbXBsYXRlOgogICAgICAgICAgICByZW5kZXJlZCA9IFRlbXBsYXRlKHRlbXBsYXRlKS5zYWZlX3N1YnN0aXR1dGUoCiAgICAgICAgICAgICAgICBqb2JJZD1qb2JfaWQsCiAgICAgICAgICAgICAgICBhdmF0YXJQYXRoPXN0cihhdmF0YXJfcGF0aCksCiAgICAgICAgICAgICAgICBhdWRpb1BhdGg9c3RyKGF1ZGlvX3BhdGgpLAogICAgICAgICAgICAgICAgb3V0cHV0UGF0aD1zdHIob3V0cHV0X3BhdGgpLAogICAgICAgICAgICAgICAgam9iRGlyPXN0cihqb2JfZGlyKSwKICAgICAgICAgICAgICAgIG1vZGVsc0Rpcj1zdHIoc2VsZi5tb2RlbHNfZGlyKSwKICAgICAgICAgICAgKQogICAgICAgICAgICByZXR1cm4gc2hsZXguc3BsaXQocmVuZGVyZWQpCgogICAgICAgIHJlcG9fcGF0aCA9IG9zLmdldGVudigiTVVTRVRBTEtfUkVQT19QQVRIIikKICAgICAgICBpZiBub3QgcmVwb19wYXRoOgogICAgICAgICAgICByYWlzZSBNdXNlVGFsa0V4ZWN1dGlvbkVycm9yKAogICAgICAgICAgICAgICAgIkNvbmZpZ3VyZSBNVVNFVEFMS19SRVBPX1BBVEggb3UgTVVTRVRBTEtfQ09NTUFORF9URU1QTEFURSBwYXJhIGFwb250YXIgcGFyYSB1bWEgaW5zdGFsYcOnw6NvIGRvIE11c2VUYWxrLiIKICAgICAgICAgICAgKQoKICAgICAgICByZXBvID0gUGF0aChyZXBvX3BhdGgpCiAgICAgICAgaW5mZXJlbmNlX3NjcmlwdCA9IFBhdGgob3MuZ2V0ZW52KCJNVVNFVEFMS19JTkZFUkVOQ0VfU0NSSVBUIiwgcmVwbyAvICJzY3JpcHRzIiAvICJpbmZlcmVuY2UucHkiKSkKICAgICAgICBpZiBub3QgaW5mZXJlbmNlX3NjcmlwdC5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgTXVzZVRhbGtFeGVjdXRpb25FcnJvcihmIlNjcmlwdCBkZSBpbmZlcsOqbmNpYSBkbyBNdXNlVGFsayBuw6NvIGVuY29udHJhZG86IHtpbmZlcmVuY2Vfc2NyaXB0fSIpCgogICAgICAgIGNvbmZpZ19wYXRoID0gc2VsZi5fd3JpdGVfaW5mZXJlbmNlX2NvbmZpZygKICAgICAgICAgICAgam9iX2lkPWpvYl9pZCwKICAgICAgICAgICAgYXZhdGFyX3BhdGg9YXZhdGFyX3BhdGgsCiAgICAgICAgICAgIGF1ZGlvX3BhdGg9YXVkaW9fcGF0aCwKICAgICAgICAgICAgam9iX2Rpcj1qb2JfZGlyLAogICAgICAgICkKICAgICAgICAKICAgICAgICBweXRob24gPSBvcy5nZXRlbnYoIk1VU0VUQUxLX1BZVEhPTiIpIG9yIHN5cy5leGVjdXRhYmxlCiAgICAgICAgdmVyc2lvbiA9IG9zLmdldGVudigiTVVTRVRBTEtfVkVSU0lPTiIsICJ2MTUiKQogICAgICAgIHVuZXRfbW9kZWxfcGF0aCA9IG9zLmdldGVudigiTVVTRVRBTEtfVU5FVF9NT0RFTF9QQVRIIiwgIm1vZGVscy9tdXNldGFsa1YxNS91bmV0LnB0aCIpCiAgICAgICAgdW5ldF9jb25maWcgPSBvcy5nZXRlbnYoIk1VU0VUQUxLX1VORVRfQ09ORklHIiwgIm1vZGVscy9tdXNldGFsa1YxNS9tdXNldGFsay5qc29uIikKICAgICAgICAKICAgICAgICByZXR1cm4gWwogICAgICAgICAgICBweXRob24sIAogICAgICAgICAgICBzdHIoaW5mZXJlbmNlX3NjcmlwdCksIAogICAgICAgICAgICAiLS1pbmZlcmVuY2VfY29uZmlnIiwgc3RyKGNvbmZpZ19wYXRoKSwKICAgICAgICAgICAgIi0tdmVyc2lvbiIsIHZlcnNpb24sCiAgICAgICAgICAgICItLXVuZXRfbW9kZWxfcGF0aCIsIHVuZXRfbW9kZWxfcGF0aCwKICAgICAgICAgICAgIi0tdW5ldF9jb25maWciLCB1bmV0X2NvbmZpZwogICAgICAgIF0KCiAgICBkZWYgX3dyaXRlX2luZmVyZW5jZV9jb25maWcoc2VsZiwgam9iX2lkOiBzdHIsIGF2YXRhcl9wYXRoOiBQYXRoLCBhdWRpb19wYXRoOiBQYXRoLCBqb2JfZGlyOiBQYXRoKSAtPiBQYXRoOgogICAgICAgICMgTXVzZVRhbGsgdXNlcyBZQU1MIGNvbmZpZ3Mgc2hhcGVkIGFzIHRhc2sgbWFwcy4gS2VlcCB0aGlzIGZpbGUgbWluaW1hbCBhbmQKICAgICAgICAjIGxldCB0aGUgY2hlY2tlZC1vdXQgTXVzZVRhbGsgcmVwb3NpdG9yeSBvd24gbW9kZWwvY2hlY2twb2ludCBzZXR0aW5ncy4KICAgICAgICBjb25maWdfcGF0aCA9IGpvYl9kaXIgLyAibXVzZXRhbGstaW5mZXJlbmNlLnlhbWwiCiAgICAgICAgY29uZmlnX3BhdGgud3JpdGVfdGV4dCgKICAgICAgICAgICAgInRhc2tfMDpcbiIKICAgICAgICAgICAgZiIgIHZpZGVvX3BhdGg6ICd7YXZhdGFyX3BhdGguYXNfcG9zaXgoKX0nXG4iCiAgICAgICAgICAgIGYiICBhdWRpb19wYXRoOiAne2F1ZGlvX3BhdGguYXNfcG9zaXgoKX0nXG4iCiAgICAgICAgICAgIGYiICByZXN1bHRfZGlyOiAne2pvYl9kaXIuYXNfcG9zaXgoKX0nXG4iCiAgICAgICAgICAgIGYiICByZXN1bHRfbmFtZTogJ3tqb2JfaWR9Lm1wNCdcbiIsCiAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgKQogICAgICAgIHJldHVybiBjb25maWdfcGF0aAoKICAgIGRlZiBfcmVzb2x2ZV9vdXRwdXQoc2VsZiwgam9iX2RpcjogUGF0aCwgZXhwZWN0ZWRfcGF0aDogUGF0aCkgLT4gUGF0aDoKICAgICAgICBpZiBleHBlY3RlZF9wYXRoLmV4aXN0cygpOgogICAgICAgICAgICByZXR1cm4gZXhwZWN0ZWRfcGF0aAoKICAgICAgICBjYW5kaWRhdGVzID0gW1BhdGgocCkgZm9yIHAgaW4gZ2xvYi5nbG9iKHN0cihqb2JfZGlyIC8gIioqIiAvICIqLm1wNCIpLCByZWN1cnNpdmU9VHJ1ZSldCiAgICAgICAgcmVwb19wYXRoID0gb3MuZ2V0ZW52KCJNVVNFVEFMS19SRVBPX1BBVEgiKQogICAgICAgIGlmIHJlcG9fcGF0aDoKICAgICAgICAgICAgcmVwb19yZXN1bHRzID0gUGF0aChyZXBvX3BhdGgpIC8gInJlc3VsdHMiCiAgICAgICAgICAgIGNhbmRpZGF0ZXMuZXh0ZW5kKFBhdGgocCkgZm9yIHAgaW4gZ2xvYi5nbG9iKHN0cihyZXBvX3Jlc3VsdHMgLyAiKioiIC8gIioubXA0IiksIHJlY3Vyc2l2ZT1UcnVlKSkKCiAgICAgICAgZXhpc3RpbmcgPSBbY2FuZGlkYXRlIGZvciBjYW5kaWRhdGUgaW4gY2FuZGlkYXRlcyBpZiBjYW5kaWRhdGUuZXhpc3RzKCldCiAgICAgICAgaWYgbm90IGV4aXN0aW5nOgogICAgICAgICAgICByYWlzZSBNdXNlVGFsa0V4ZWN1dGlvbkVycm9yKCJNdXNlVGFsayB0ZXJtaW5vdSwgbWFzIG5lbmh1bSBhcnF1aXZvIC5tcDQgZGUgc2HDrWRhIGZvaSBlbmNvbnRyYWRvLiIpCgogICAgICAgIHByZWZlcnJlZCA9IFtjYW5kaWRhdGUgZm9yIGNhbmRpZGF0ZSBpbiBleGlzdGluZyBpZiBjYW5kaWRhdGUubmFtZSA9PSBmIntleHBlY3RlZF9wYXRoLnBhcmVudC5uYW1lfS5tcDQiXQogICAgICAgIG5vbl90ZW1wID0gW2NhbmRpZGF0ZSBmb3IgY2FuZGlkYXRlIGluIGV4aXN0aW5nIGlmIG5vdCBjYW5kaWRhdGUubmFtZS5zdGFydHN3aXRoKCJ0ZW1wXyIpXQogICAgICAgIHBvb2wgPSBwcmVmZXJyZWQgb3Igbm9uX3RlbXAgb3IgZXhpc3RpbmcKICAgICAgICBsYXRlc3QgPSBtYXgocG9vbCwga2V5PWxhbWJkYSBwYXRoOiBwYXRoLnN0YXQoKS5zdF9tdGltZSkKICAgICAgICBpZiBsYXRlc3QgIT0gZXhwZWN0ZWRfcGF0aDoKICAgICAgICAgICAgc2h1dGlsLmNvcHkyKGxhdGVzdCwgZXhwZWN0ZWRfcGF0aCkKICAgICAgICByZXR1cm4gZXhwZWN0ZWRfcGF0aAo="

app_path = WORK_DIR / 'app.py'
service_path = WORK_DIR / 'musetalk_service.py'

app_path.write_bytes(base64.b64decode(app_b64))
service_path.write_bytes(base64.b64decode(service_b64))

print('Arquivos escritos com sucesso:')
print(' -', app_path)
print(' -', service_path)


In [ ]:
# 6. Iniciar FastAPI e expor via Cloudflare Tunnel
import os, queue, re, stat, threading, subprocess, time, requests, sys
PYTHON_EXE = os.environ.get('PYTHON') or sys.executable
os.environ['MPLBACKEND'] = 'Agg'

cloudflared = WORK_DIR / 'cloudflared'
if not cloudflared.exists():
    print("Baixando binário do Cloudflare Tunnel...")
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', str(cloudflared)], check=True)
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

# Finaliza instâncias anteriores para liberar porta se o notebook for reiniciado
for name in ['server_proc', 'tunnel_proc']:
    proc = globals().get(name)
    if proc and proc.poll() is None:
        print(f"Finalizando processo antigo: {name}")
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()

print("Iniciando serviço FastAPI com Uvicorn...")
os.environ['PYTHONPATH'] = f"{str(WORK_DIR)}:{os.environ.get('PYTHONPATH', '')}"
server_proc = subprocess.Popen(
    [PYTHON_EXE, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', str(PORT), '--proxy-headers'],
    cwd=str(WORK_DIR),
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

def stream_logs(prefix, proc):
    for line in proc.stdout:
        print(prefix, line, end='', flush=True)
threading.Thread(target=stream_logs, args=('[uvicorn]', server_proc), daemon=True).start()

time.sleep(5)
try:
    resp = requests.get(f'http://127.0.0.1:{PORT}/health', timeout=15)
    print('Smoke Test Health Local:', resp.status_code, resp.json())
except Exception as e:
    print('Erro no smoke test local de saúde:', e)

print("Iniciando túnel Cloudflare...")
tunnel_proc = subprocess.Popen(
    [str(cloudflared), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

q = queue.Queue()
def capture_tunnel():
    for line in tunnel_proc.stdout:
        print('[cloudflared]', line, end='', flush=True)
        q.put(line)
threading.Thread(target=capture_tunnel, daemon=True).start()

PUBLIC_URL = None
pattern = re.compile(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com')
deadline = time.time() + 90
while time.time() < deadline and PUBLIC_URL is None:
    try:
        line = q.get(timeout=2)
    except queue.Empty:
        continue
    match = pattern.search(line)
    if match:
        PUBLIC_URL = match.group(0)
if not PUBLIC_URL:
    raise RuntimeError('Não consegui obter URL pública do cloudflared.')

print('\n=== CONFIGURE NO .env.local DO MRCHICKEN ===')
print('LIPSYNC_ENGINE=musetalk-v15')
print(f'LIPSYNC_API_URL={PUBLIC_URL}')
print(f'LIPSYNC_API_KEY={API_KEY}')
print('LIPSYNC_TRANSFER_MODE=upload')
print('LIPSYNC_TIMEOUT_MS=1800000')
print('\nEndpoint público:', PUBLIC_URL)
